# Spark DataFrame 조회

- **T (Transformation)**: `select`, `filter`, `orderBy`, `groupBy`, `agg`, `limit` 등. 새 DataFrame을 반환하고 실제 계산은 미룸 (Lazy)
- **A (Action)**: `show`, `count`, `head`, `collect`, `toPandas` 등. 이때 실제 계산이 실행됨
- Spark DataFrame은 수정 불가(immutable). 변환할 때마다 새 DataFrame이 만들어짐

## 0. Setup

In [173]:
import pandas as pd
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F  # F.count, F.max, F.upper ...
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
        .appName("SparkML")
        .enableHiveSupport()
        .getOrCreate()
)

DATA_PATH = Path("./data")

26/10/15 08:33:54 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. 데이터 로드

| 코드 | 종류 | 설명 |
|---|---|---|
| `spark.read.csv(..., header=True, inferSchema=True)` | - | inferSchema: 컬럼 타입 자동 추론 (데이터를 한 번 더 읽음) |
| `cache()` | T | 반복 사용할 DataFrame을 메모리에 보관. 첫 Action 때 실제로 저장됨 |
| `toPandas()` | A | 전체 데이터를 Driver 한 곳으로 모음. **큰 데이터에서는 메모리 터짐 주의** |

In [174]:
titanic_sdf = spark.read.csv(str(DATA_PATH / "titanic_train.csv"), header=True, inferSchema=True)
titanic_sdf = titanic_sdf.cache()     # T
titanic_pdf = titanic_sdf.toPandas()  # A: 작은 데이터라 pandas 비교용으로 변환

print(type(titanic_sdf))
print(type(titanic_pdf))

<class 'pyspark.sql.dataframe.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


26/10/15 08:33:56 WARN CacheManager: Asked to cache already cached data.


## 2. 데이터 훑어보기

| 목적 | pandas | Spark | Spark 반환 타입 |
|---|---|---|---|
| 앞부분 출력 | `head(n)` | `show(n)` (A) | None (출력만 함) |
| 앞 n개 자르기 | `head(n)` | `limit(n)` (T) | DataFrame |
| 앞 n개 가져오기 | `head(n)` | `head(n)` (A) | list[Row] |
| 구조 | `info()` | `printSchema()` | None |
| 크기 | `shape` | `count()`, `len(columns)` | int |
| 기술통계 | `describe()` | `describe().show()` | DataFrame |

- `print(df.show())`처럼 쓰면 show가 반환한 None까지 찍히므로 `show()`만 단독으로 사용
- `describe()`는 문자열 컬럼(Name 등)도 포함: count/min/max는 사전순으로 계산되고 mean/stddev는 null

In [175]:
titanic_sdf.show(5)                     # A: 출력만 하고 None 반환

print(type(titanic_sdf.limit(5)))       # T: DataFrame -> 보려면 .show() 필요

rows = titanic_sdf.head(2)              # A: Row 객체의 list
print(type(rows), rows[0]["Name"])

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [177]:
titanic_sdf.printSchema()                     # pandas: info()

print(f"row 수: {titanic_sdf.count()}")       # pandas: shape[0]
print(f"col 수: {len(titanic_sdf.columns)}")  # pandas: shape[1]

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

row 수: 891
col 수: 12


In [178]:
# 컬럼별 결측치 개수 (pandas: titanic_pdf.isna().sum())
# isnan: 숫자형 NaN, isNull: null
titanic_sdf.select([
    F.count(F.when(F.isnan(c) | col(c).isNull(), c)).alias(c)
    for c in titanic_sdf.columns
]).show()

+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+
|PassengerId|Survived|Pclass|Name|Sex|Age|SibSp|Parch|Ticket|Fare|Cabin|Embarked|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+
|          0|       0|     0|   0|  0|177|    0|    0|     0|   0|  687|       2|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+



In [179]:
# 숫자형 컬럼만 골라서 기술통계
num_cols = [c for c, t in titanic_sdf.dtypes if t != "string"]
titanic_sdf.select(*num_cols).describe().show()

+-------+-----------------+-------------------+------------------+------------------+------------------+-------------------+-----------------+
|summary|      PassengerId|           Survived|            Pclass|               Age|             SibSp|              Parch|             Fare|
+-------+-----------------+-------------------+------------------+------------------+------------------+-------------------+-----------------+
|  count|              891|                891|               891|               714|               891|                891|              891|
|   mean|            446.0| 0.3838383838383838| 2.308641975308642| 29.69911764705882|0.5230078563411896|0.38159371492704824| 32.2042079685746|
| stddev|257.3538420152301|0.48659245426485753|0.8360712409770491|14.526497332334035|1.1027434322934315| 0.8060572211299488|49.69342859718089|
|    min|                1|                  0|                 1|              0.42|                 0|                  0|              0.0|

## 3. 컬럼 선택 (select) - T

- 컬럼을 가리키는 방법 4가지: `"Name"`, `df.Name`, `df["Name"]`, `col("Name")` (앞 3개는 결과 동일, `col()`은 DataFrame 이름 없이 쓸 수 있어서 체이닝할 때 편함)
- 여러 컬럼을 list로 넣을 땐 `select(*cols)`처럼 unpack 권장
- **alias는 Column에 붙여야 컬럼 이름이 바뀜**. DataFrame에 붙이면(`df.count().alias(...)`) DataFrame 별칭이 됨

In [180]:
ex_dict = {
    "Name": ["Haebin", "Chulsoo", "Wansoo", "Minchul", "Yoori"],
    "Year": [1999, 2001, 2003, 2002, 1998],
    "Gender": ["Male", "Male", "Male", "Female", "Female"]
}
data_pdf = pd.DataFrame(ex_dict)
data_sdf = spark.createDataFrame(data_pdf)
data_sdf.show()

+-------+----+------+
|   Name|Year|Gender|
+-------+----+------+
| Haebin|1999|  Male|
|Chulsoo|2001|  Male|
| Wansoo|2003|  Male|
|Minchul|2002|Female|
|  Yoori|1998|Female|
+-------+----+------+



In [183]:
# 4가지 컬럼 참조 방식을 섞어 써도 동작
data_sdf.select("Name", data_sdf.Year, data_sdf["Gender"]).show()
data_sdf.select(col("Name")).show()

# 조건으로 컬럼 고르기 + unpack
str_cols = [c for c, t in data_sdf.dtypes if t == "string"]
data_sdf.select(*str_cols).show()

+-------+----+------+
|   Name|Year|Gender|
+-------+----+------+
| Haebin|1999|  Male|
|Chulsoo|2001|  Male|
| Wansoo|2003|  Male|
|Minchul|2002|Female|
|  Yoori|1998|Female|
+-------+----+------+

+-------+
|   Name|
+-------+
| Haebin|
|Chulsoo|
| Wansoo|
|Minchul|
|  Yoori|
+-------+

+-------+------+
|   Name|Gender|
+-------+------+
| Haebin|  Male|
|Chulsoo|  Male|
| Wansoo|  Male|
|Minchul|Female|
|  Yoori|Female|
+-------+------+



In [184]:
# 함수 적용 + alias로 컬럼 이름 지정
data_sdf.select("*", F.upper(col("Name")).alias("Cap_name")).show()

+-------+----+------+--------+
|   Name|Year|Gender|Cap_name|
+-------+----+------+--------+
| Haebin|1999|  Male|  HAEBIN|
|Chulsoo|2001|  Male| CHULSOO|
| Wansoo|2003|  Male|  WANSOO|
|Minchul|2002|Female| MINCHUL|
|  Yoori|1998|Female|   YOORI|
+-------+----+------+--------+



## 4. 행 필터링 (filter) - T

| 방식 | 예시 |
|---|---|
| Column 조건 | `filter(col("Name") == "Haebin")` |
| SQL 문자열 | `filter("Name == 'Haebin'")` |
| AND / OR | `filter((조건1) & (조건2))`, `filter((조건1) | (조건2))` **각 조건 괄호 필수** |
| 패턴 | `col("Name").like("%soo")`, `"Name like '%soo'"` |

- 오류 예: `filter("Name" == "Haebin")` -> 파이썬이 문자열 두 개를 먼저 비교해서 False를 넘겨버림
- `where()`는 `filter()`와 같은 함수

In [185]:
data_sdf.filter(col("Name") == "Haebin").show()   # Column 조건
data_sdf.filter("Name == 'Haebin'").show()        # SQL 문자열 조건
data_sdf.filter("Name" == "Haebin").show() # 오류

+------+----+------+
|  Name|Year|Gender|
+------+----+------+
|Haebin|1999|  Male|
+------+----+------+

+------+----+------+
|  Name|Year|Gender|
+------+----+------+
|Haebin|1999|  Male|
+------+----+------+



PySparkTypeError: [NOT_COLUMN_OR_STR] Argument `condition` should be a Column or str, got bool.

In [186]:
data_sdf.filter((col("Gender") == "Male") & (col("Year") >= 2000)).show()  # AND
data_sdf.filter((col("Gender") == "Male") | (col("Year") >= 2000)).show()  # OR

data_sdf.filter(col("Gender") == "Male" & col("Year") >= 2000).show()  # 오류: 괄호 생략 -> &가 ==, >=보다 먼저 계산됨

+-------+----+------+
|   Name|Year|Gender|
+-------+----+------+
|Chulsoo|2001|  Male|
| Wansoo|2003|  Male|
+-------+----+------+

+-------+----+------+
|   Name|Year|Gender|
+-------+----+------+
| Haebin|1999|  Male|
|Chulsoo|2001|  Male|
| Wansoo|2003|  Male|
|Minchul|2002|Female|
+-------+----+------+



Py4JError: An error occurred while calling o1750.and. Trace:
py4j.Py4JException: Method and([class java.lang.String]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:321)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:329)
	at py4j.Gateway.invoke(Gateway.java:274)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.base/java.lang.Thread.run(Thread.java:829)



In [187]:
data_sdf.filter(col("Name").like("%soo")).show()
data_sdf.filter("upper(Name) like '%SOO'").show()

# filter -> select 
data_sdf.filter(F.upper(col("Name")).like("%SOO")).select("Name", "Gender").show()

+-------+----+------+
|   Name|Year|Gender|
+-------+----+------+
|Chulsoo|2001|  Male|
| Wansoo|2003|  Male|
+-------+----+------+

+-------+----+------+
|   Name|Year|Gender|
+-------+----+------+
|Chulsoo|2001|  Male|
| Wansoo|2003|  Male|
+-------+----+------+

+-------+------+
|   Name|Gender|
+-------+------+
|Chulsoo|  Male|
| Wansoo|  Male|
+-------+------+



## 5. 정렬 (orderBy) - T

| pandas | Spark |
|---|---|
| `sort_values(by=["A"], ascending=True)` | `orderBy(col("A"), ascending=True)` |
| `sort_values(by=["A", "B"], ascending=[True, False])` | `orderBy(col("A").asc(), col("B").desc())` |

- 컬럼마다 방향이 다르면 `asc()` / `desc()`가 읽기 쉬움
- Spark는 데이터가 나뉘어 있어서 **orderBy 없이는 행 순서가 보장되지 않음** (pandas처럼 index도 없음)

In [188]:
# SQL: select Pclass, Name from titanic order by Pclass asc, Name desc
titanic_sdf.orderBy(col("Pclass").asc(), col("Name").desc()).select("Pclass", "Name").show(5)

# pandas 비교
titanic_pdf.sort_values(by=["Pclass", "Name"], ascending=[True, False])[["Pclass", "Name"]].head()

+------+--------------------+
|Pclass|                Name|
+------+--------------------+
|     1|Young, Miss. Mari...|
|     1|  Wright, Mr. George|
|     1|   Woolner, Mr. Hugh|
|     1|Williams-Lambert,...|
|     1|Williams, Mr. Cha...|
+------+--------------------+
only showing top 5 rows



,Pclass,Name
325,1,"Young, Miss. Marie Grice"
555,1,"Wright, Mr. George"
55,1,"Woolner, Mr. Hugh"
351,1,"Williams-Lambert, Mr. Fletcher Fellows"
155,1,"Williams, Mr. Charles Duane"


## 6. 집계 (count / agg / groupBy)

- DataFrame에 바로 쓸 수 있는 집계 메서드는 사실상 `count()`뿐 (A, int 반환). `titanic_sdf.max()`는 오류
- 나머지는 `F.max()` 같은 함수를 `select()`나 `agg()` 안에 넣어서 사용 -> 결과도 DataFrame
- `groupBy().count()`의 결과 컬럼명은 `count`. 이름을 바꾸려면 `agg(F.count("*").alias(...))`
- 집계 결과에 `filter`를 걸면 SQL의 HAVING과 같음

| pandas | Spark |
|---|---|
| `df.count()` (컬럼별 개수, Series) | `df.count()` (전체 행 수, int) |
| `groupby("A")["A"].count()` | `groupBy("A").count()` |
| `groupby("A").agg({"B": "max"})` | `groupBy("A").agg(F.max("B").alias("max_b"))` |

In [189]:
print(titanic_sdf.count())                                   # A: int
titanic_sdf.select(F.max("Age"), F.avg("Fare")).show()       # DataFrame으로 반환
titanic_sdf.max() # max 메서드는 없음. 바로 aggregation을 할 수 있는 메서드가 매우 드뭄

891
+--------+----------------+
|max(Age)|       avg(Fare)|
+--------+----------------+
|    80.0|32.2042079685746|
+--------+----------------+



AttributeError: 'DataFrame' object has no attribute 'max'

In [191]:
# 그룹별 개수 + 이름 지정 + 정렬
titanic_sdf.groupBy("Pclass").agg(F.count("*").alias("count_n")).orderBy(col("count_n").desc()).show()

# 여러 컬럼으로 그룹핑 (간단 집계 메서드)
titanic_sdf.groupBy("Pclass", "Sex").max("Age").orderBy("Pclass", "Sex").show()

+------+-------+
|Pclass|count_n|
+------+-------+
|     3|    491|
|     1|    216|
|     2|    184|
+------+-------+

+------+------+--------+
|Pclass|   Sex|max(Age)|
+------+------+--------+
|     1|female|    63.0|
|     1|  male|    80.0|
|     2|female|    57.0|
|     2|  male|    70.0|
|     3|female|    63.0|
|     3|  male|    74.0|
+------+------+--------+



In [192]:
# 여러 집계를 한 번에 + 집계 결과로 필터 (HAVING)
(
    titanic_sdf.groupBy("Pclass")
    .agg(
        F.max("Age").alias("max_age"),
        F.min("Age").alias("min_age"),
        F.sum("Age").alias("sum_age"),
        F.avg("Age").alias("avg_age"),
    )
    .filter(col("avg_age") > 26)
    .orderBy("Pclass")
    .show()
)

# pandas 비교
titanic_pdf.groupby("Pclass").agg({"Age": "max", "SibSp": "sum", "Fare": "mean"})

+------+-------+-------+-------+------------------+
|Pclass|max_age|min_age|sum_age|           avg_age|
+------+-------+-------+-------+------------------+
|     1|   80.0|   0.92|7111.42|38.233440860215055|
|     2|   70.0|   0.67|5168.83| 29.87763005780347|
+------+-------+-------+-------+------------------+



,Age,SibSp,Fare
Pclass,,,
1,80.0,90,84.154687
2,70.0,74,20.662183
3,74.0,302,13.675550
